# 01 · Fair-value engine

**Story:** the exchange IOPV is stale during A-share hours because the US underlying is closed, so we reconstruct fair value ourselves and trade the premium to *that*, not to the stale number.

> Runs on the **synthetic** sample; numbers are illustrative. The real validation figure is in the top-level README.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

import qdii_fve

ROOT = Path(qdii_fve.__file__).resolve().parents[2]
SAMPLE = ROOT / "data" / "sample"

bars = pd.read_csv(SAMPLE / "SYNTHETIC_daily_bars.csv", parse_dates=["date"])
nav = pd.read_csv(SAMPLE / "SYNTHETIC_nav.csv", parse_dates=["date"])
und = pd.read_csv(SAMPLE / "SYNTHETIC_ndx_underlying.csv", parse_dates=["date"])
und = und.set_index("date")["close"]
fx = pd.read_csv(SAMPLE / "SYNTHETIC_usdcnh.csv", parse_dates=["date"]).set_index("date")

close = bars.pivot(index="date", columns="etf_id", values="close")
bid = bars.pivot(index="date", columns="etf_id", values="bid")
ask = bars.pivot(index="date", columns="etf_id", values="ask")
nav_pub = nav.pivot(index="date", columns="etf_id", values="nav").reindex(close.index)
print("SYNTHETIC sample — illustrative only, not the real figures.")
print(f"{close.shape[1]} funds, {close.shape[0]} days")

## Reconstruct fair value

`FV(t) = NAV0 · U(t)/U(d0) · FX(t)/FX(d0)` — the last published NAV rolled forward by the NASDAQ-100 and USDCNH move since the NAV date (see [`docs/fair_value_derivation.md`](../docs/fair_value_derivation.md)).

In [ ]:
from qdii_fve.fair_value.reconstruct import reconstruct_series

# NAV here is one session lagged, so its strike date is the previous day: scale
# the published NAV by the underlying and USDCNH move since then. (The library's
# roll_forward_nav handles arbitrary vintages; this sample uses a 1-day lag.)
u_prev = und.shift(1)
fx_prev = fx["usdcnh"].shift(1)
fair_value = pd.DataFrame(index=close.index, columns=close.columns, dtype=float)
for f in close.columns:
    fair_value[f] = reconstruct_series(
        nav_value=nav_pub[f],
        nav_underlying=u_prev,
        nav_fx=fx_prev,
        current_underlying=und,
        current_fx=fx["usdcnh"],
    )
fair_value.head()

## Price vs reconstructed fair value

The gap between the traded price and fair value is the **premium** — the tradable quantity. Against the stale IOPV this gap would be mismeasured.

In [ ]:
f = close.columns[0]
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(close.index, close[f], label=f"{f} price", lw=1)
ax.plot(fair_value.index, fair_value[f], label=f"{f} reconstructed fair value", lw=1)
ax.set_title(f"{f}: price vs reconstructed fair value (synthetic)")
ax.legend()
plt.tight_layout()

## The premium panel is the signal input

Premium in basis points for every fund; the cross-sectional dispersion is what the relative-value signal trades.

In [ ]:
from qdii_fve.backtest.signal import observed_premium_bps

premium_bps = observed_premium_bps(close, fair_value)
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(premium_bps.index, premium_bps.values, lw=0.7, alpha=0.7)
ax.axhline(0, color="k", lw=0.5)
ax.set_title("Premium to fair value, bp (synthetic, 12 funds)")
ax.set_ylabel("bp")
plt.tight_layout()

## Validation: computed fair value vs realised NAV

When the fund later publishes the NAV a reconstruction was predicting, the residual measures model error. On this idealised synthetic data the model is near-exact (residual ≈ rounding). On **real** data the residual standard deviation is a few bp — the exact figure is in the README — arising from tracking error, NAV timing, and FX-reference frictions the sample omits.

In [ ]:
from qdii_fve.fair_value.reconstruct import validation_residuals_bps

realised = nav_pub.shift(-1)  # the NAV that FV(t) predicts is published next session
res = pd.concat(
    [validation_residuals_bps(fair_value[f], realised[f]) for f in close.columns]
)
print(f"synthetic residual: std = {res.std():.2f} bp, MAE = {res.abs().mean():.2f} bp")
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(res.dropna(), bins=40)
ax.set_title("Fair-value residual vs realised NAV (synthetic, bp)")
ax.set_xlabel("bp")
plt.tight_layout()

## Takeaways

- Fair value is reconstructed from NAV × underlying × FX, not read from a stale IOPV.
- The **premium to fair value** is the quantity every later stage uses.
- The method is validated against realised NAV (README for the real figure).

Next: [`02_bounce_decomposition`](02_bounce_decomposition.ipynb) — the microstructure honesty check.